# **Proyecto de Pandas**

Primero ejecutamos la libreria pandas que es la que utilizaremos durante nuestro proyecto.

In [2]:
import pandas as pd

In [8]:
df = pd.read_csv("25001.txt", sep=r"\s+", engine="python")
df.to_csv("25001.csv", index=False)

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xd3 in position 264: invalid continuation byte

Debido a que el archivo no esta codificado con UTF-8 lo cual crea un error, utilizamos la libreria chardet 
para saber con que esta codificado.

In [9]:
import chardet

with open("25001.txt", "rb") as f:
    resultado = chardet.detect(f.read())
print(resultado)  

{'encoding': 'cp1250', 'confidence': 0.8336787254899339, 'language': 'sr', 'mime_type': 'text/plain'}


nos muestra que esta en cp2150

In [10]:
df = pd.read_csv("25001.txt", sep=r"\s+", engine="python", encoding="cp1252")
df.to_csv("25001.csv", index=False)

ParserError: Expected 4 fields in line 6, saw 17. Error could possibly be due to quotes being ignored when a multi-char delimiter is used.

**El error que nos dio ahora significa que no es un solo espacio en blanco el delimitador, por lo cual inspeccionaremos el archivo crudo para ver su organizacion**

In [13]:
with open("25001.txt", "r", encoding="cp1252") as f:
    for i, linea in enumerate(f):
        print(repr(linea))
        if i > 20:  # muestra las primeras 20 líneas
            break

'CNA-SMN-CG-GMC-SMAA-CLIMATOLOGIA\n'
'BASE DE DATOS CLIMATOLOGICA\n'
'DATOS DISPONIBLES EN LA BASE DE DATOS A MARZO 2020;CON LA INFORMACION SUMINISTRADA POR LAS OFICINAS REGIONALES\n'
' \n'
'ESTACION  : 25001\n'
'NOMBRE    : ACATITAN\n'
'ESTADO    : SINALOA\n'
'MUNICIPIO : SAN IGNACIO\n'
'SITUACIÓN : OPERANDO\n'
'ORGANISMO : CONAGUA-DGE\n'
'CVE-OMM   : Nulo\n'
'LATITUD   : 024.097°\n'
'LONGITUD  : -106.667°\n'
'ALTITUD   : 96 msnm\n'
' \n'
'EMISION   : 06/04/2020 \n'
' \n'
'           PRECIP  EVAP   TMAX   TMIN\n'
'  FECHA     (MM)   (MM)   (°C)   (°C)\n'
'01/01/1961  0     Nulo    25     13 \n'
'02/01/1961  0     Nulo    27.5   14 \n'
'03/01/1961  0     Nulo    31     11 \n'


**Al ya saber el orden y la manera de resolverlo, eliminaremos los datos innecesarios y eliminaremos las columnas, para posteriormente agregarlas manualmente, esto para conservar las "medidas" en las que se realizaron los datos**

In [3]:
df = pd.read_csv(
    '25001.txt', #Nombre del archivo
    sep=r"\s+", #Separado mediante espacios 
    engine="python", #Motor de análisis
    encoding="cp1250", #Codificación del archivo
    skiprows=17, #Saltar "metadatos"
    header=None, #No hay encabezado en el archivo
    names = ["FECHA", "PRECIP (MM)", 'EVAP (MM)', 'TMAX (°C)', 'TMIN (°C)'] #Nombres de las columnas
    )

#Eliminamos las dos primeras columnas ya que son los encabezados originales del txt
df = df.iloc[2:].reset_index(drop=True)

df.to_csv("25001.csv", index=False) #Mediante este comando se guarda el DataFrame en un nuevo archivo CSV
print(df.head())

        FECHA PRECIP (MM) EVAP (MM) TMAX (°C) TMIN (°C)
0  01/01/1961           0      Nulo        25        13
1  02/01/1961           0      Nulo      27.5        14
2  03/01/1961           0      Nulo        31        11
3  04/01/1961           0      Nulo        32        13
4  05/01/1961           0      Nulo        29        14


De esta manera ya tenemos nuestro archivo txt convertido a csv

# **1.- Eliminar rellenar nulos con la media de la columna** 

In [4]:
#Importamos la libreria numpy para realizar la operacion que nos piden
import numpy as np


**Pandas reconoce los 'Nulo' que se encuentran en nuestro csv como datos string, por lo cual primero debemos de convertirlo a NaN para que ahora si los tome como vacio**

In [30]:
df.replace('Nulo', np.nan, inplace=True) #Reemplazamos los valores "Nulo" por NaN 

,FECHA,PRECIP (MM),EVAP (MM),TMAX (°C),TMIN (°C)
0,01/01/1961,0,NaN,25,13
1,02/01/1961,0,NaN,27.5,14
2,03/01/1961,0,NaN,31,11
3,04/01/1961,0,NaN,32,13
4,05/01/1961,0,NaN,29,14
...,...,...,...,...,...
19570,28/07/2018,9.8,3.7,37,23.5
19571,29/07/2018,6,3.3,35,20
19572,30/07/2018,48,6,36,22
19573,31/07/2018,2,3.7,34,22


**Ahora debemos de convertir las columnas numericas a float, ya que se encuentran en tipo objeto y no es posible calcular la media**

In [31]:
columnas = [' PRECIP (MM)', ' EVAP (MM)', ' TMAX (°C)', ' TMIN (°C)'] 
df[columnas] = df[columnas].astype(float) #Convertimos las columnas a tipo float 


**Ahora resolveremos lo pedido, rellenar los nulos con la media de la columna**

In [32]:
df[columnas] = df[columnas].fillna(df[columnas].mean())

In [33]:
df.head()

,FECHA,PRECIP (MM),EVAP (MM),TMAX (°C),TMIN (°C)
0,01/01/1961,0.0,4.882157,25.0,13.0
1,02/01/1961,0.0,4.882157,27.5,14.0
2,03/01/1961,0.0,4.882157,31.0,11.0
3,04/01/1961,0.0,4.882157,32.0,13.0
4,05/01/1961,0.0,4.882157,29.0,14.0


# **2.- Contar total de registros faltantes por fecha**

**Volvemos a cargar el archivo original, ya que el NaN afecto en la columna de fechas**

In [51]:
df = pd.read_csv('25001.csv')
df.head()

,FECHA,PRECIP (MM),EVAP (MM),TMAX (°C),TMIN (°C)
0,01/01/1961,0,Nulo,25,13
1,02/01/1961,0,Nulo,27.5,14
2,03/01/1961,0,Nulo,31,11
3,04/01/1961,0,Nulo,32,13
4,05/01/1961,0,Nulo,29,14


In [52]:

df.replace('Nulo', np.nan, inplace=True) #Reemplazamos los valores "Nulo" por NaN 
columnas = [' PRECIP (MM)', ' EVAP (MM)', ' TMAX (°C)', ' TMIN (°C)'] 
df[columnas] = df[columnas].astype(float) #Convertimos las columnas a tipo float 

In [ ]:
total_faltantes = 0
for i in range(len(df)): #Recorremos cada fila del DataFrame
    faltantes_fila = 0
    for col in columnas: #Recorre cada columna dentro de esa fila
        if df[col][i] != df[col][i]:  # NaN no es igual a sí mismo
            faltantes_fila += 1
    
    if faltantes_fila > 0:
        print(f"Fecha: {df['FECHA'][i]} Faltantes: {faltantes_fila}") #Aqui imprime la fecha y el numero de faltantes
        total_faltantes += 1

print(f"\nTotal de fechas con al menos un faltante: {total_faltantes}")

Fecha: 01/01/1961 Faltantes: 1
Fecha: 02/01/1961 Faltantes: 1
Fecha: 03/01/1961 Faltantes: 1
Fecha: 04/01/1961 Faltantes: 1
Fecha: 05/01/1961 Faltantes: 1
Fecha: 06/01/1961 Faltantes: 1
Fecha: 07/01/1961 Faltantes: 1
Fecha: 08/01/1961 Faltantes: 1
Fecha: 09/01/1961 Faltantes: 1
Fecha: 10/01/1961 Faltantes: 1
Fecha: 11/01/1961 Faltantes: 1
Fecha: 12/01/1961 Faltantes: 1
Fecha: 13/01/1961 Faltantes: 1
Fecha: 14/01/1961 Faltantes: 1
Fecha: 15/01/1961 Faltantes: 1
Fecha: 16/01/1961 Faltantes: 1
Fecha: 17/01/1961 Faltantes: 1
Fecha: 18/01/1961 Faltantes: 1
Fecha: 19/01/1961 Faltantes: 1
Fecha: 20/01/1961 Faltantes: 1
Fecha: 21/01/1961 Faltantes: 1
Fecha: 22/01/1961 Faltantes: 1
Fecha: 23/01/1961 Faltantes: 1
Fecha: 24/01/1961 Faltantes: 1
Fecha: 25/01/1961 Faltantes: 1
Fecha: 26/01/1961 Faltantes: 1
Fecha: 27/01/1961 Faltantes: 1
Fecha: 28/01/1961 Faltantes: 1
Fecha: 29/01/1961 Faltantes: 1
Fecha: 30/01/1961 Faltantes: 1
Fecha: 31/01/1961 Faltantes: 1
Fecha: 01/02/1961 Faltantes: 1
Fecha: 0

# **Trabajo modulado a partir de aqui**

In [10]:
def cargar_datos():
    df=pd.read_csv(
        '25001.txt', #Nombre del archivo
        sep=r"\s+", #Separado mediante espacios 
        engine="python", #Motor de análisis
        encoding="cp1250", #Codificación del archivo
        skiprows=17, #Saltar "metadatos"
        header=None, #No hay encabezado en el archivo
        names = ["FECHA", "PRECIP (MM)", 'EVAP (MM)', 'TMAX (°C)', 'TMIN (°C)'] #Nombres de las columnas
    )
    #Eliminamos las dos primeras columnas ya que son los encabezados originales del txt
    df = df.iloc[2:].reset_index(drop=True)

    df.to_csv("25001.csv", index=False) #Mediante este comando se guarda el DataFrame en un nuevo archivo CSV
    return df

In [11]:
df = cargar_datos()
df

,FECHA,PRECIP (MM),EVAP (MM),TMAX (°C),TMIN (°C)
0,01/01/1961,0,Nulo,25,13
1,02/01/1961,0,Nulo,27.5,14
2,03/01/1961,0,Nulo,31,11
3,04/01/1961,0,Nulo,32,13
4,05/01/1961,0,Nulo,29,14
...,...,...,...,...,...
19570,28/07/2018,9.8,3.7,37,23.5
19571,29/07/2018,6,3.3,35,20
19572,30/07/2018,48,6,36,22
19573,31/07/2018,2,3.7,34,22


In [20]:
def limpiar(df):
    columnas = ['PRECIP (MM)', 'EVAP (MM)', 'TMAX (°C)', 'TMIN (°C)']
    df.replace('Nulo', np.nan, inplace=True)
    df[columnas] = df[columnas].astype(float)
    return df
    

In [23]:
df = limpiar(df)
df.head()

,FECHA,PRECIP (MM),EVAP (MM),TMAX (°C),TMIN (°C)
0,01/01/1961,0.0,NaN,25.0,13.0
1,02/01/1961,0.0,NaN,27.5,14.0
2,03/01/1961,0.0,NaN,31.0,11.0
3,04/01/1961,0.0,NaN,32.0,13.0
4,05/01/1961,0.0,NaN,29.0,14.0


In [24]:
def rellenar_NaN(df):
    columnas = ['PRECIP (MM)', 'EVAP (MM)', 'TMAX (°C)', 'TMIN (°C)']
    df[columnas] = df[columnas].fillna(df[columnas].mean())
    return df

In [25]:
df = rellenar_NaN(df)
df.head()

,FECHA,PRECIP (MM),EVAP (MM),TMAX (°C),TMIN (°C)
0,01/01/1961,0.0,4.882157,25.0,13.0
1,02/01/1961,0.0,4.882157,27.5,14.0
2,03/01/1961,0.0,4.882157,31.0,11.0
3,04/01/1961,0.0,4.882157,32.0,13.0
4,05/01/1961,0.0,4.882157,29.0,14.0


In [46]:
def faltantes(df):
    columnas = ['PRECIP (MM)', 'EVAP (MM)', 'TMAX (°C)', 'TMIN (°C)']
    total_faltantes = 0
    for i in range(len(df)):
        faltantes_fila = 0
        for col in columnas:
            if df[col][i] != df[col][i]:
                faltantes_fila += 1
        if faltantes_fila > 0:
            print(f"Fecha: {df['FECHA'][i]} Faltantes: {faltantes_fila}")
            total_faltantes += 1
    print(f"\nTotal de fechas con al menos un faltante: {total_faltantes}")       

In [48]:
df = cargar_datos()
df = limpiar(df)
faltantes(df)

Fecha: 01/01/1961 Faltantes: 1
Fecha: 02/01/1961 Faltantes: 1
Fecha: 03/01/1961 Faltantes: 1
Fecha: 04/01/1961 Faltantes: 1
Fecha: 05/01/1961 Faltantes: 1
Fecha: 06/01/1961 Faltantes: 1
Fecha: 07/01/1961 Faltantes: 1
Fecha: 08/01/1961 Faltantes: 1
Fecha: 09/01/1961 Faltantes: 1
Fecha: 10/01/1961 Faltantes: 1
Fecha: 11/01/1961 Faltantes: 1
Fecha: 12/01/1961 Faltantes: 1
Fecha: 13/01/1961 Faltantes: 1
Fecha: 14/01/1961 Faltantes: 1
Fecha: 15/01/1961 Faltantes: 1
Fecha: 16/01/1961 Faltantes: 1
Fecha: 17/01/1961 Faltantes: 1
Fecha: 18/01/1961 Faltantes: 1
Fecha: 19/01/1961 Faltantes: 1
Fecha: 20/01/1961 Faltantes: 1
Fecha: 21/01/1961 Faltantes: 1
Fecha: 22/01/1961 Faltantes: 1
Fecha: 23/01/1961 Faltantes: 1
Fecha: 24/01/1961 Faltantes: 1
Fecha: 25/01/1961 Faltantes: 1
Fecha: 26/01/1961 Faltantes: 1
Fecha: 27/01/1961 Faltantes: 1
Fecha: 28/01/1961 Faltantes: 1
Fecha: 29/01/1961 Faltantes: 1
Fecha: 30/01/1961 Faltantes: 1
Fecha: 31/01/1961 Faltantes: 1
Fecha: 01/02/1961 Faltantes: 1
Fecha: 0

In [61]:
import os #Modulo patra limpiar la pantalla despues de cada opcion del menu

def limpiar_pantalla():
    os.system('cls' if os.name == 'nt' else 'clear')

def menu():
    df = None  # El DataFrame empieza vacío
    
    while True:
        print("   MENÚ - PROCESAMIENTO DE DATOS CLIMÁTICOS")
        print("1. Cargar datos")
        print("2. Limpiar nulos")
        print("3. Contar faltantes")
        print("4. Rellenar con media")
        print("5. Salir")
        
        opcion = input("Selecciona una opción: ").strip()

        if opcion == "1":
            df = cargar_datos()
            print(df.head())
            input('\nPulsa enter para continuar')

        elif opcion == "2":
            if df is None:
                print("Primero debes cargar los datos ")
            else:
                df = limpiar(df)
                print(df.head())
                input('\nPulsa enter para continuar')

        elif opcion == "3":
            if df is None:
                print("Primero debes cargar los datos")
            else:
                faltantes(df)
                print(df.head())
                input('\nPulsa enter para continuar')

        elif opcion == "4":
            if df is None:
                print("Primero debes cargar los datos ")
            else:
                df = rellenar_NaN(df)
                print(" Nulos rellenados con la media")
                print(df.head())
                input('\nPulsa enter para continuar')

        elif opcion == "5":
            limpiar_pantalla()
            print("Nos vemos luego")
            break

In [ ]:
menu()

   MENÚ - PROCESAMIENTO DE DATOS CLIMÁTICOS
1. Cargar datos
2. Limpiar nulos
3. Contar faltantes
4. Rellenar con media
5. Salir
Nos vemos luego
